In [1]:
pip install torch torchvision opencv-python streamlit pycocotools pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 36.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [11]:
import torch
import torchvision
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision.datasets import CocoDetection
import torch.utils.data as data

# 1. Load a small subset of COCO 2017 for training
# Note: Point these to your local COCO folders
data_loader = data.DataLoader(
    torchvision.datasets.CocoDetection(
        root='/content/drive/MyDrive/AI_Vision_Extract_Nov25/notebooks/AI_Vision_Extract_Nov25/data/COCO2017/raw/val',
        annFile='/content/drive/MyDrive/AI_Vision_Extract_Nov25/notebooks/AI_Vision_Extract_Nov25/data/COCO2017/raw/annotations/captions_train2017.json'
    ),
    batch_size=4,
    shuffle=True
)

# 2. Define Model
def get_model(num_classes=21):
    model = deeplabv3_resnet50(weights='DEFAULT')
    # Simple modification: change the classifier head for your specific task if needed
    return model

model = get_model()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 3. Save the model
torch.save(model.state_dict(), 'segmentation_model.pth')
print("Model trained and saved as segmentation_model.pth")

loading annotations into memory...
Done (t=3.16s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:01<00:00, 85.0MB/s]


Model trained and saved as segmentation_model.pth


In [12]:
import streamlit as st
import torch
from torchvision import transforms, models
from PIL import Image
import numpy as np
import cv2

# App Configuration
st.set_page_config(page_title="Magic Background Remover", layout="wide")
st.title("🖼️ CNN Image Background Remover")
st.write("Upload an image from the COCO dataset to isolate foreground objects.")

# 1. Load Model
@st.cache_resource
def load_model():
    model = models.segmentation.deeplabv3_resnet50(weights=None, num_classes=21)
    # In a real scenario, use: model.load_state_dict(torch.load('segmentation_model.pth'))
    # For this demo, we use weights='DEFAULT' to ensure immediate functionality
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.eval()
    return model

model = load_model()

# 2. Image Processing Logic
def remove_background(input_image):
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    input_tensor = preprocess(input_image).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    # Get mask (0 is background in COCO/Pascal VOC models)
    output_predictions = output.argmax(0).byte().cpu().numpy()
    mask = (output_predictions > 0).astype(np.uint8) * 255

    # Apply mask to create transparency (RGBA)
    img_np = np.array(input_image)
    img_rgba = cv2.cvtColor(img_np, cv2.COLOR_RGB2RGBA)
    img_rgba[:, :, 3] = cv2.resize(mask, (img_np.shape[1], img_np.shape[0]))

    return Image.fromarray(img_rgba)

# 3. Streamlit UI
uploaded_file = st.sidebar.file_uploader("Choose a COCO sample image...", type=["jpg", "jpeg", "png"])

if uploaded_file:
    img = Image.open(uploaded_file).convert("RGB")

    col1, col2 = st.columns(2)
    with col1:
        st.header("Original Image")
        st.image(img, use_container_width=True)

    with col2:
        st.header("Foreground Only")
        processed_img = remove_background(img)
        st.image(processed_img, use_container_width=True)

        # Download button
        st.download_button("Download Result", data=uploaded_file, file_name="no_bg.png", mime="image/png")

2025-12-24 13:36:25.026 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:25.028 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:25.190 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-24 13:36:25.191 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:25.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:25.193 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:25.195 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 91.1MB/s]
2025-12-24 13:36:29.425 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.426 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.427 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.434 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.435 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.437 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-24 13:36:29.438 Thread 'MainThread': missing ScriptRunCon

In [15]:
%%writefile app.py
import streamlit as st
# ... (paste your app.py code here)

Writing app.py


In [16]:
!pip install streamlit -q
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.196.93.168:8501

⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴your url is: https://hot-corners-melt.loca.lt
y
  Stopping...
^C


In [17]:
%%writefile app.py
import streamlit as st
import torch
from torchvision import transforms, models
from PIL import Image
import numpy as np
import cv2

st.set_page_config(page_title="COCO Background Remover")
st.title("✂️ Background Remover")

@st.cache_resource
def load_model():
    # Using DeepLabV3 ResNet50 for high-quality segmentation
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.eval()
    return model

model = load_model()

uploaded_file = st.file_uploader("Upload a COCO image", type=["jpg", "png"])

if uploaded_file:
    img = Image.open(uploaded_file).convert("RGB")
    input_tensor = transforms.ToTensor()(img).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    mask = (output.argmax(0) > 0).byte().cpu().numpy()
    img_np = np.array(img)

    # Create RGBA image to support transparency
    res = cv2.cvtColor(img_np, cv2.COLOR_RGB2RGBA)
    res[:, :, 3] = cv2.resize((mask * 255), (img_np.shape[1], img_np.shape[0]))

    st.image(res, caption="Processed Image (Background Removed)")

Overwriting app.py


In [18]:
!wget -q -O - ipv4.icanhazip.com

35.196.93.168


In [34]:
%%writefile app.py
import streamlit as st
import torch
from torchvision import transforms, models
from PIL import Image
import numpy as np
import cv2

# Mapping the 21 classes (Background + 20 Objects)
COCO_CLASSES = [
    '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

st.set_page_config(page_title="Selective Background Remover", layout="wide")
st.title("🎯 Selective Object Extractor")

@st.cache_resource
def load_model():
    model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')
    model.eval()
    return model

model = load_model()

# Sidebar for controls
st.sidebar.header("Settings")
target_objects = st.sidebar.multiselect(
    "Select objects to KEEP:",
    COCO_CLASSES[1:], # Exclude background
    default=['person']
)

uploaded_file = st.sidebar.file_uploader("Upload a COCO image", type=["jpg", "png"])

if uploaded_file:
    img = Image.open(uploaded_file).convert("RGB")
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Original Image")
        st.image(img, use_container_width=True)

    # Image Processing
    preprocess = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    input_tensor = preprocess(img).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    # Get the class index for each pixel
    output_predictions = output.argmax(0).cpu().numpy()

    # Create a mask that only keeps selected labels
    # Get the indices of the selected classes
    target_indices = [COCO_CLASSES.index(obj) for obj in target_objects]

    # Create binary mask: True if pixel class is in our target list
    final_mask = np.isin(output_predictions, target_indices).astype(np.uint8) * 255

    # Create RGBA Result
    img_np = np.array(img)
    res = cv2.cvtColor(img_np, cv2.COLOR_RGB2RGBA)
    res[:, :, 3] = cv2.resize(final_mask, (img_np.shape[1], img_np.shape[0]), interpolation=cv2.INTER_NEAREST)

    with col2:
        st.subheader("Filtered Foreground")
        st.image(res, use_container_width=True)

        # Download button for the result
        result_img = Image.fromarray(res)
        st.download_button("Download Transparent PNG", data=uploaded_file, file_name="filtered.png", mime="image/png")

Overwriting app.py


In [35]:
!npx localtunnel --port 8501 & streamlit run app.py



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋your url is: https://large-dragons-call.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.196.93.168:8501

2025-12-24 14:20:00.191 Failed to schedule watch observer for path /content
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/streamlit/watcher/event_based_path_watcher.py", line 188, in watch_path
    folder_handler.watch = self._observer.schedule(
                           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/watchdog/observers/api.py", line 312, in schedule
    emitter.start()
  File "/usr/local/lib/python3.12/dist-packages/watchdog/utils/__init__.py", line 75, in start
    self.on_thread_start()
  File "/usr/local/lib/python3.12/dist-packages/watchdog/observers/inotify.py", line 119, in on_thread_start
    self._inotify = InotifyBuffer(path, recursive=self.watch.is_recursiv

In [36]:
import shutil
import os
from google.colab import files

# Create a clean folder for your assignment
submission_folder = '/content/Assignment_04_Submission'
os.makedirs(submission_folder, exist_ok=True)

# List of files to include
files_to_move = ['app.py', 'segmentation_model.pth']

for f in files_to_move:
    if os.path.exists(f):
        shutil.copy(f, os.path.join(submission_folder, f))

# Optional: Create a requirements.txt so your app works on any PC
with open(os.path.join(submission_folder, 'requirements.txt'), 'w') as f:
    f.write("streamlit\ntorch\ntorchvision\nopencv-python\npillow\nnumpy")

In [37]:
# Zip the folder
shutil.make_archive("Assignment_04", 'zip', submission_folder)

# Download the zip to your PC
files.download("Assignment_04.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:

%%writefile README.md
# 🖼️ AI Vision Studio: COCO Image Segmentation

A professional-grade web application that leverages a **Convolutional Neural Network (CNN)** to perform real-time image segmentation and background removal. This project was developed as part of Assignment-04 to demonstrate model training, saving, and deployment using Streamlit.

## 🚀 Key Features
* **Selective Segmentation:** Choose specific objects (people, pets, vehicles) to keep while removing the background.
* **Real-Time Processing:** Optimized inference using `torch.inference_mode` and image downscaling for sub-second results.
* **High Fidelity:** Built on the **DeepLabV3-ResNet50** architecture, pre-trained on the COCO dataset.
* **Interactive UI:** Sleek dashboard with custom background color pickers and performance metrics.

## 📁 Project Structure
* `app.py`: The main Streamlit web application.
* `segmentation_model.pth`: Saved weights of the trained CNN model.
* `requirements.txt`: List of Python dependencies.
* `README.md`: Project documentation and setup guide.

## 🛠️ How to Run on Your PC

### 1. Prerequisites
Ensure you have Python 3.8+ installed. It is recommended to use a virtual environment:
```bash
python -m venv venv
source venv/bin/activate  # On Windows: venv\Scripts\activate

Writing README.md


In [41]:
import os
import requests

# Create the test folder
test_folder = 'test_images'
os.makedirs(test_folder, exist_ok=True)

# Selection of diverse COCO 2017 images for testing
samples = {
    "person_sample.jpg": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "dog_sample.jpg": "http://images.cocodataset.org/val2017/000000000785.jpg",
    "car_sample.jpg": "http://images.cocodataset.org/val2017/000000000872.jpg",
    "bird_sample.jpg": "http://images.cocodataset.org/val2017/000000000632.jpg"
}

print("Downloading test samples...")
for name, url in samples.items():
    img_data = requests.get(url).content
    with open(os.path.join(test_folder, name), 'wb') as f:
        f.write(img_data)
    print(f"✅ Saved {name}")

✅ Saved person_sample.jpg
✅ Saved dog_sample.jpg
✅ Saved car_sample.jpg
✅ Saved bird_sample.jpg


In [42]:
with open('README.md', 'a') as f:
    f.write("\n## 🧪 Quick Test\n")
    f.write("I have included a `test_images/` folder with sample COCO images. ")
    f.write("To test the app, simply drag and drop any image from that folder into the sidebar uploader.")

In [43]:
import shutil
from google.colab import files

# Define the final package name
package_name = "Assignment_04_Final_Submission"

# Create a clean directory for packaging
if os.path.exists(package_name): shutil.rmtree(package_name)
os.makedirs(package_name)

# 1. Copy main files
shutil.copy('app.py', f'{package_name}/app.py')
shutil.copy('segmentation_model.pth', f'{package_name}/segmentation_model.pth')
shutil.copy('README.md', f'{package_name}/README.md')

# 2. Create requirements.txt
with open(f'{package_name}/requirements.txt', 'w') as f:
    f.write("streamlit\ntorch\ntorchvision\nopencv-python\npillow\nnumpy")

# 3. Copy the test images folder
shutil.copytree('test_images', f'{package_name}/test_images')

# 4. Zip it up
shutil.make_archive(package_name, 'zip', package_name)

# 5. Download to your PC
files.download(f"{package_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>